## English

In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import json
import re

SECTION_KEYWORDS = [
    (("reading 1", "first reading", "reading i"), "reading1"),
    (("responsorial psalm", "psalm"), "psalm"),
    (("reading 2", "second reading", "reading ii"), "reading2"),
    (("alleluia", "gospel acclamation", "verse before the gospel"), "alleluia"),
    (("gospel",), "gospel"),
]

def _looks_like_citation(text):
    return bool(re.match(
        r'^(cf\.\s*)?[1-3]?\s*[A-Z][a-zA-Z]+\.?\s+\d+(:\d+)?([-,]\s*\d+)*[a-z]?$',
        text.strip()
    ))

def _parse_usccb_html(html_content):
    eng_dict = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
    soup = BeautifulSoup(html_content, 'html.parser')

    current_section = None
    current_text = []

    def save_current_option():
        if not current_section or not current_text:
            return
        full_text = "\n".join(current_text).strip()
        if full_text and full_text not in eng_dict[current_section]:
            eng_dict[current_section].append(full_text)

    for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'div', 'a', 'span']):
        text = tag.get_text(separator=' ', strip=True)
        if not text:
            continue
        lower_text = text.lower().strip()

        # Stop harvesting completely if we hit the page footer elements
        if any(footer_trigger in lower_text for footer_trigger in [
            "email terms & privacy", "united states conference of catholic bishops", 
            "listen podcast", "en español", "view calendar", "get daily readings",
            "lectionary for mass", "privacy policy", "usccb", "subscribe"
        ]):
            if current_section == "gospel":
                save_current_option()
                current_section = None
            continue

        is_heading_tag = tag.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']

        if is_heading_tag and lower_text == "or":
            save_current_option()
            current_text = []
            continue

        if is_heading_tag or len(text) < 40:
            matched = None
            for keys, section in SECTION_KEYWORDS:
                if any(k in lower_text for k in keys):
                    matched = section
                    break
            if matched:
                save_current_option()
                current_section = matched
                current_text = []
                continue

        if current_section:
            if _looks_like_citation(text):
                continue
            if tag.name in ['p', 'div'] and not tag.find(['div', 'p']):
                if text not in current_text:
                    current_text.append(text)

    save_current_option()
    return eng_dict

async def scrape_usccb_async(date_str):
    url = f"https://bible.usccb.org/bible/readings/{date_str}.cfm"
    empty = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            )
        )
        try:
            print(f"  -> Navigating to USCCB for {date_str}...")
            await page.goto(url, wait_until="networkidle", timeout=15000)
            html_content = await page.content()
        except Exception as e:
            print(f"  -> Error loading page: {e}")
            return empty
        finally:
            await browser.close()

    return _parse_usccb_html(html_content)

In [3]:
target_date = "072626"
print(f"Initializing Async Playwright Scraper for USCCB ({target_date})...")
readings = await scrape_usccb_async(target_date)

print("\n" + "=" * 50)
print("USCCB ENGLISH READINGS PAYLOAD")
print("=" * 50)
print(json.dumps(readings, indent=4, ensure_ascii=False))

Initializing Async Playwright Scraper for USCCB (072626)...
  -> Navigating to USCCB for 072626...

USCCB ENGLISH READINGS PAYLOAD
{
    "reading1": [
        "The LORD appeared to Solomon in a dream at night. God said, \"Ask something of me and I will give it to you.\" Solomon answered: \"O LORD, my God, you have made me, your servant, king to succeed my father David; but I am a mere youth, not knowing at all how to act. I serve you in the midst of the people whom you have chosen, a people so vast that it cannot be numbered or counted. Give your servant, therefore, an understanding heart to judge your people and to distinguish right from wrong. For who is able to govern this vast people of yours?\" The LORD was pleased that Solomon made this request. So God said to him: \"Because you have asked for this— not for a long life for yourself, nor for riches, nor for the life of your enemies, but for understanding so that you may know what is right— I do as you requested. I give you a heart

## Vietnamese

In [9]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime

def scrape_lavang(url):
    """
    Scrapes the Vietnamese Mass readings from a static La Vang Church URL.
    Uses heuristic keyword matching to group text into the correct sections.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'
    }
    
    viet_dict = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = 'utf-8' 
    except requests.exceptions.RequestException as e:
        print(f"Error fetching La Vang data: {e}")
        return viet_dict
        
    soup = BeautifulSoup(response.content, 'html.parser')
    
    current_section = None
    current_text = []
    
    for tag in soup.find_all(['p', 'h3', 'h4', 'b', 'strong', 'div', 'span']):
        text = tag.get_text(separator='\n', strip=True)
        # Clean up weird liturgical characters often found in Missals
        text = text.replace('✠', '').replace('Xướng:', '').strip()
        
        if not text:
            continue
            
        lower_text = text.lower()
        
        # Determine if we need to switch sections based on the text
        if any(x in lower_text for x in ["bài đọc 1", "bài đọc i", "bài trích", "thư thứ nhất"]) and current_section != "reading1":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "reading1"
            current_text = []
            
        # Added "đ./" to trigger the switch to the Psalm section
        elif any(x in lower_text for x in ["đáp ca", "thánh vịnh", "đ./"]) and current_section != "psalm":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "psalm"
            current_text = []
            
        elif any(x in lower_text for x in ["bài đọc 2", "bài đọc ii"]) and current_section != "reading2":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "reading2"
            current_text = []
            
        elif any(x in lower_text for x in ["alleluia", "tung hô"]) and current_section != "alleluia":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "alleluia"
            current_text = []
            
        elif any(x in lower_text for x in ["tin mừng", "phúc âm"]) and current_section != "gospel":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "gospel"
            current_text = []
            
        # Append the text to the current active section bucket
        if current_section:
            current_text.append(text)
            
    # Append the final accumulated section
    if current_section and current_text:
        viet_dict[current_section].append("\n".join(current_text))
        
    return viet_dict

def prepare_template_data(user_inputs, scraped_eng, scraped_viet):
    """
    Filters the scraped data dictionaries based on the user's language 
    and option preferences for each specific reading.
    """
    final_data = {"eng": {}, "viet": {}}
    
    for section, choices in user_inputs.items():
        if section in ["date", "lavang_url"]:
            continue
            
        lang = choices.get("lang", "none")
        idx = choices.get("option_index", 0)
        
        # English Population
        if lang in ["eng", "bilingual"]:
            if section in scraped_eng and len(scraped_eng[section]) > idx:
                final_data["eng"][section] = scraped_eng[section][idx]
            else:
                final_data["eng"][section] = None
        else:
            final_data["eng"][section] = None
                
        # Vietnamese Population
        if lang in ["viet", "bilingual"]:
            if section in scraped_viet and len(scraped_viet[section]) > idx:
                final_data["viet"][section] = scraped_viet[section][idx]
            else:
                final_data["viet"][section] = None
        else:
            final_data["viet"][section] = None
                
    return final_data

In [10]:

if __name__ == "__main__":
    app_inputs = {
        "date": "072526",
        "lavang_url": "https://lavangchurch.org/wp-content/uploads/DailyReadings/FeastDay/0725_James.html",
        "reading1": {"lang": "bilingual", "option_index": 0},
        "psalm":    {"lang": "bilingual", "option_index": 0},
        "reading2": {"lang": "none",      "option_index": 0}, 
        "alleluia": {"lang": "viet",      "option_index": 0},
        "gospel":   {"lang": "bilingual", "option_index": 0}  
    }
    
    print("Scraping USCCB...")
    eng_readings = scrape_usccb_async(app_inputs["date"])
    
    print("Scraping La Vang...")
    viet_readings = scrape_lavang(app_inputs["lavang_url"])
    
    print("Merging Data Based on User Inputs...")
    final_document_data = prepare_template_data(app_inputs, eng_readings, viet_readings)
    
    print("\n" + "="*60)
    print("FINAL DATA PAYLOAD FOR DOCUMENT GENERATION")
    print("="*60)
    print(json.dumps(final_document_data, indent=4, ensure_ascii=False))

Scraping USCCB...
Scraping La Vang...
Merging Data Based on User Inputs...


TypeError: argument of type 'coroutine' is not iterable

---
---

In [12]:
import asyncio
import json
import re
import requests
from bs4 import BeautifulSoup

# ==========================================
# 1. ENGLISH SCRAPER (USCCB - Playwright)
# ==========================================

SECTION_KEYWORDS = [
    (("reading 1", "first reading", "reading i"), "reading1"),
    (("responsorial psalm", "psalm"), "psalm"),
    (("reading 2", "second reading", "reading ii"), "reading2"),
    (("alleluia", "gospel acclamation", "verse before the gospel"), "alleluia"),
    (("gospel",), "gospel"),
]

def _looks_like_citation(text):
    return bool(re.match(
        r'^(cf\.\s*)?[1-3]?\s*[A-Z][a-zA-Z]+\.?\s+\d+(:\d+)?([-,]\s*\d+)*[a-z]?$',
        text.strip()
    ))

def _parse_usccb_html(html_content):
    eng_dict = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
    soup = BeautifulSoup(html_content, 'html.parser')

    current_section = None
    current_text = []

    def save_current_option():
        if not current_section or not current_text:
            return
        full_text = "\n".join(current_text).strip()
        if full_text and full_text not in eng_dict[current_section]:
            eng_dict[current_section].append(full_text)

    for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'div', 'a', 'span']):
        text = tag.get_text(separator=' ', strip=True)
        if not text:
            continue
        lower_text = text.lower().strip()

        if any(footer_trigger in lower_text for footer_trigger in [
            "email terms & privacy", "united states conference of catholic bishops", 
            "listen podcast", "en español", "view calendar", "get daily readings",
            "lectionary for mass", "privacy policy", "usccb", "subscribe"
        ]):
            if current_section == "gospel":
                save_current_option()
                current_section = None
            continue

        is_heading_tag = tag.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']

        if is_heading_tag and lower_text == "or":
            save_current_option()
            current_text = []
            continue

        if is_heading_tag or len(text) < 40:
            matched = None
            for keys, section in SECTION_KEYWORDS:
                if any(k in lower_text for k in keys):
                    matched = section
                    break
            if matched:
                save_current_option()
                current_section = matched
                current_text = []
                continue

        if current_section:
            if _looks_like_citation(text):
                continue
            if tag.name in ['p', 'div'] and not tag.find(['div', 'p']):
                if text not in current_text:
                    current_text.append(text)

    save_current_option()
    return eng_dict

async def scrape_usccb_async(date_str):
    from playwright.async_api import async_playwright
    url = f"https://bible.usccb.org/bible/readings/{date_str}.cfm"
    empty = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            )
        )
        try:
            print(f"  -> Navigating to USCCB for {date_str}...")
            await page.goto(url, wait_until="networkidle", timeout=15000)
            html_content = await page.content()
        except Exception as e:
            print(f"  -> Error loading page: {e}")
            return empty
        finally:
            await browser.close()

    return _parse_usccb_html(html_content)


# ==========================================
# 2. VIETNAMESE SCRAPER (La Vang - Requests)
# ==========================================

def scrape_lavang(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'
    }
    
    viet_dict = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = 'utf-8' 
    except requests.exceptions.RequestException as e:
        print(f"Error fetching La Vang data: {e}")
        return viet_dict
        
    soup = BeautifulSoup(response.content, 'html.parser')
    
    current_section = None
    current_text = []
    
    for tag in soup.find_all(['p', 'h3', 'h4', 'b', 'strong', 'div', 'span']):
        text = tag.get_text(separator='\n', strip=True)
        text = text.replace('✠', '').replace('Xướng:', '').strip()
        
        if not text:
            continue
            
        lower_text = text.lower()
        
        if any(x in lower_text for x in ["bài đọc 1", "bài đọc i", "bài trích", "thư thứ nhất"]) and current_section != "reading1":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "reading1"
            current_text = []
            
        elif any(x in lower_text for x in ["đáp ca", "thánh vịnh", "đ./"]) and current_section != "psalm":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "psalm"
            current_text = []
            
        elif any(x in lower_text for x in ["bài đọc 2", "bài đọc ii"]) and current_section != "reading2":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "reading2"
            current_text = []
            
        elif any(x in lower_text for x in ["alleluia", "tung hô"]) and current_section != "alleluia":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "alleluia"
            current_text = []
            
        elif any(x in lower_text for x in ["tin mừng", "phúc âm"]) and current_section != "gospel":
            if current_section and current_text:
                viet_dict[current_section].append("\n".join(current_text))
            current_section = "gospel"
            current_text = []
            
        if current_section:
            current_text.append(text)
            
    if current_section and current_text:
        viet_dict[current_section].append("\n".join(current_text))
        
    return viet_dict


# ==========================================
# 3. DATA MERGING & TEMPLATE MAPPING
# ==========================================

def prepare_template_data(user_inputs, scraped_eng, scraped_viet):
    final_data = {"eng": {}, "viet": {}}
    
    for section, choices in user_inputs.items():
        if section in ["date", "lavang_url"]:
            continue
            
        lang = choices.get("lang", "none")
        idx = choices.get("option_index", 0)
        
        # English Population
        if lang in ["eng", "bilingual"]:
            if section in scraped_eng and len(scraped_eng[section]) > idx:
                final_data["eng"][section] = scraped_eng[section][idx]
            else:
                final_data["eng"][section] = None
        else:
            final_data["eng"][section] = None
                
        # Vietnamese Population
        if lang in ["viet", "bilingual"]:
            if section in scraped_viet and len(scraped_viet[section]) > idx:
                final_data["viet"][section] = scraped_viet[section][idx]
            else:
                final_data["viet"][section] = None
        else:
            final_data["viet"][section] = None
                
    return final_data


# ==========================================
# 4. MAIN EXECUTION ENTRY POINT
# ==========================================

async def main():
    app_inputs = {
        "date": "072526",
        "lavang_url": "https://lavangchurch.org/wp-content/uploads/DailyReadings/FeastDay/0725_James.html",
        "reading1": {"lang": "bilingual", "option_index": 0},
        "psalm":    {"lang": "bilingual", "option_index": 0},
        "reading2": {"lang": "none",      "option_index": 0}, 
        "alleluia": {"lang": "viet",      "option_index": 0},
        "gospel":   {"lang": "bilingual", "option_index": 0}    
    }
    
    print("Scraping USCCB (English)...")
    eng_readings = await scrape_usccb_async(app_inputs["date"])
    
    print("Scraping La Vang (Vietnamese)...")
    viet_readings = scrape_lavang(app_inputs["lavang_url"])
    
    print("Merging Data Based on User Inputs...")
    final_document_data = prepare_template_data(app_inputs, eng_readings, viet_readings)
    
    print("\n" + "="*60)
    print("FINAL DATA PAYLOAD FOR DOCUMENT GENERATION")
    print("="*60)
    print(json.dumps(final_document_data, indent=4, ensure_ascii=False))

# ==========================================
# 4. MAIN EXECUTION ENTRY POINT (Jupyter / Colab Safe)
# ==========================================

async def main():
    app_inputs = {
        "date": "072526",
        "lavang_url": "https://lavangchurch.org/wp-content/uploads/DailyReadings/FeastDay/0725_James.html",
        "reading1": {"lang": "bilingual", "option_index": 0},
        "psalm":    {"lang": "bilingual", "option_index": 0},
        "reading2": {"lang": "none",      "option_index": 0}, 
        "alleluia": {"lang": "viet",      "option_index": 0},
        "gospel":   {"lang": "bilingual", "option_index": 0}    
    }
    
    print("Scraping USCCB (English)...")
    eng_readings = await scrape_usccb_async(app_inputs["date"])
    
    print("Scraping La Vang (Vietnamese)...")
    viet_readings = scrape_lavang(app_inputs["lavang_url"])
    
    print("Merging Data Based on User Inputs...")
    final_document_data = prepare_template_data(app_inputs, eng_readings, viet_readings)
    
    print("\n" + "="*60)
    print("FINAL DATA PAYLOAD FOR DOCUMENT GENERATION")
    print("="*60)
    print(json.dumps(final_document_data, indent=4, ensure_ascii=False))

# Run directly using await for Jupyter/IPython environments
await main()

Scraping USCCB (English)...
  -> Navigating to USCCB for 072526...
Scraping La Vang (Vietnamese)...
Merging Data Based on User Inputs...

FINAL DATA PAYLOAD FOR DOCUMENT GENERATION
{
    "eng": {
        "reading1": "Brothers and sisters: We hold this treasure in earthen vessels, that the surpassing power may be of God and not from us. We are afflicted in every way, but not constrained; perplexed, but not driven to despair; persecuted, but not abandoned; struck down, but not destroyed; always carrying about in the body the dying of Jesus, so that the life of Jesus may also be manifested in our body. For we who live are constantly being given up to death for the sake of Jesus, so that the life of Jesus may be manifested in our mortal flesh. So death is at work in us, but life in you. Since, then, we have the same spirit of faith, according to what is written, I believed, therefore I spoke, we too believe and therefore speak, knowing that the one who raised the Lord Jesus will raise us a